# CVDD — resultados e diagnóstico

Este notebook é **somente de leitura de artefatos**: não treina nem sobrescreve CVDD. Ele usa as métricas e os gráficos gerados em `experiment_results/unsupervised/text_reports/cvdd/`.

Se o diretório ainda não existir no cluster, gere-o uma vez com:

```bash
bash backend/slurm/submit_text_model_reports.sh cvdd
```

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, Markdown, display

cwd = Path.cwd().resolve()
candidates = [cwd / 'backend', cwd, *cwd.parents]
BACKEND = next((path for path in candidates if path.name == 'backend'), None)
if BACKEND is None:
    raise FileNotFoundError('Abra o notebook a partir do repositório ou de backend/.')
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

MODEL = 'cvdd'
REPORT_DIR = BACKEND / 'experiment_results' / 'unsupervised' / 'text_reports' / MODEL
METRICS_PATH = REPORT_DIR / 'metrics_by_split.csv'
REPORT_DIR

In [ ]:
if not METRICS_PATH.exists():
    display(Markdown('**Ainda não há relatório cacheado.** Rode `bash backend/slurm/submit_text_model_reports.sh cvdd` no cluster e reabra este notebook.'))
else:
    metrics = pd.read_csv(METRICS_PATH)
    columns = [
        'split_label', 'accuracy', 'f1_macro', 'f1_scam', 'precision_scam',
        'recall_scam', 'roc_auc', 'pr_auc', 'TP', 'TN', 'FP', 'FN',
    ]
    display(metrics[[column for column in columns if column in metrics.columns]].style.format({
        name: '{:.4f}' for name in ['accuracy', 'f1_macro', 'f1_scam', 'precision_scam', 'recall_scam', 'roc_auc', 'pr_auc']
    }))

## Como ler

- **TP/TN/FP/FN** são contagens brutas de conversas: golpes detectados corretamente, conversas legítimas corretas, alertas falsos e golpes perdidos.
- **PR-AUC** resume a separação de golpes quando a classe é desbalanceada; **ROC-AUC** mede a ordenação global dos scores.
- Compare principalmente teste interno e validação externa: diferença grande indica mudança de distribuição, não uma melhoria ou piora causada pelo notebook.
- CVDD aprende a descrição de normalidade (Ham) com vetores de contexto e atenção sobre tokens; score alto significa texto mais distante dessa normalidade.

In [ ]:
plot_names = [
    'metrics_by_split.png', 'confusion_counts_by_split.png',
    'confusion_matrix_external.png', 'roc_external.png',
    'pr_external.png', 'score_distribution_external.png',
]
for plot_name in plot_names:
    path = REPORT_DIR / 'plots' / plot_name
    if path.exists():
        display(Markdown(f'### {plot_name.replace("_", " ").replace(".png", "").title()}'))
        display(Image(filename=str(path)))
    else:
        print(f'Gráfico ainda ausente: {path.name}')